# Your Title Here

**Name(s)**: Samuel Yu, William Zhang

**Website Link**: [Spotify Analysis](https://wyllz.github.io/spotify-analysis/)

In [48]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [49]:
DATA_DIR = Path("Data")

artists_raw = pd.read_csv(DATA_DIR / "artists.csv")
tracks_raw = pd.read_csv(DATA_DIR / "music_tracks.csv")

artists_raw.shape, tracks_raw.shape

((1162095, 5), (114000, 22))

## Step 2: Data Cleaning and Exploratory Data Analysis

In [50]:
artists_raw

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0
...,...,...,...,...,...
1162090,3cOzi726Iav1toV2LRVEjp,4831.0,['black comedy'],Ali Siddiq,34
1162091,6LogY6VMM3jgAE6fPzXeMl,46.0,[],Rodney Laney,2
1162092,19boQkDEIay9GaVAWkUhTa,257.0,[],Blake Wexler,10
1162093,5nvjpU3Y7L6Hpe54QuvDjy,2357.0,['black comedy'],Donnell Rawlings,15


In [51]:
tracks_raw

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,NaN,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,2000-04,False,0.172,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,NaN,5,world-music
113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,1973,False,0.174,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,1992-10-21,False,0.629,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,1992,False,0.587,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


In [53]:
from ast import literal_eval

artists_clean = (
    artists_raw
    .rename(columns={
        "id": "artist_id",
        "name": "artist_name",
        "popularity": "artist_popularity",
        "genres": "artist_genres",
    })
    .dropna(subset=["artist_name"])
    .assign(
        artist_name=lambda df: df["artist_name"].str.strip(),
        followers=lambda df: df["followers"].fillna(0).astype(int),
        artist_genres=lambda df: df["artist_genres"].apply(
            lambda value: literal_eval(value) if isinstance(value, str) and value.startswith("[") else []
        ),
    )
    .sort_values(["followers", "artist_popularity"], ascending=False)
    .drop_duplicates(subset=["artist_name"], keep="first")
)

tracks_clean = (
    tracks_raw
    .drop(columns=["Unnamed: 0"], errors="ignore")
    .dropna(subset=["track_id", "artists", "album_name", "track_name"])
    .rename(columns={"popularity": "track_popularity"})
    .assign(
        duration_min=lambda df: df["duration_ms"] / 60000,
        release_date_parsed=lambda df: pd.to_datetime(df["release_date"], errors="coerce"),
        release_year=lambda df: pd.to_numeric(df["release_date"].str[:4], errors="coerce").astype("Int64"),
        artists_list=lambda df: df["artists"].str.split(";"),
    )
)

tracks_clean["tempo"] = tracks_clean["tempo"].fillna(
    tracks_clean.groupby("track_genre")["tempo"].transform("median")
)
tracks_clean["tempo"] = tracks_clean["tempo"].fillna(tracks_clean["tempo"].median())

tracks_artists = (
    tracks_clean
    .explode("artists_list")
    .rename(columns={"artists_list": "artist_name"})
)
tracks_artists["artist_name"] = tracks_artists["artist_name"].str.strip()

artists_tracks_df = tracks_artists.merge(
    artists_clean,
    on="artist_name",
    how="left",
    validate="many_to_one",
)

unique_tracks_df = artists_tracks_df.drop_duplicates(subset=['track_id'])
unique_tracks_df


,track_id,artists,album_name,track_name,track_popularity,duration_ms,release_date,explicit,danceability,energy,...,time_signature,track_genre,duration_min,release_date_parsed,release_year,artist_name,artist_id,followers,artist_genres,artist_popularity
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.676,0.4610,...,4,acoustic,3.844433,1974-01-01,1974,Gen Hoshino,1S2S00lgLYLGHWA44qGEUs,852637.0,"[j-acoustic, j-pop, j-rock, japanese singer-so...",66.0
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.420,0.1660,...,4,acoustic,2.493500,NaT,1995,Ben Woodward,142VT1MtWzaD13CnOiKFDn,11874.0,"[acoustic chill, viral pop]",53.0
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.438,0.3590,...,4,acoustic,3.513767,1973-01-01,1973,Ingrid Michaelson,2vm8GdHyrJh2O2MfbQFYG0,722496.0,"[acoustic pop, ectofolk, indiecoustica, lilith...",68.0
4,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.266,0.0596,...,3,acoustic,3.365550,NaT,2018,Kina Grannis,7h4j9YTJJuAHzLCc3KCvYu,438860.0,"[acoustic pop, indie cafe pop, neo mellow, vir...",71.0
5,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.618,0.4430,...,4,acoustic,3.314217,NaT,2017,Chord Overstreet,5D3muNJhYYunbRkh3FKgX0,99345.0,[acoustic pop],70.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158287,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,2000-04,False,0.172,0.2350,...,5,world-music,6.416650,NaT,2000,Rainy Lullaby,3IygNfufoCZx61wd14t19E,101.0,"[lullaby, musica de fondo]",25.0
158288,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,1973,False,0.174,0.1170,...,4,world-music,6.416667,1973-01-01,1973,Rainy Lullaby,3IygNfufoCZx61wd14t19E,101.0,"[lullaby, musica de fondo]",25.0
158289,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,1992-10-21,False,0.629,0.3290,...,4,world-music,4.524433,NaT,1992,Cesária Evora,0Nks3cFWU2a7rooAlFQYgn,247002.0,"[afropop, cape verdean folk, morna, musica cab...",59.0
158290,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,1992,False,0.587,0.5060,...,4,world-music,4.731550,1992-01-01,1992,Michael W. Smith,5aBxFPaaGk9204ssHUvXWN,353983.0,"[ccm, christian alternative rock, christian mu...",64.0


In [35]:
fig_popularity = px.histogram(
    tracks_clean,
    x="track_popularity",
    nbins=30,
    title="Distribution of Track Popularity",
    labels={"track_popularity": "Track Popularity"},
)
fig_popularity.update_layout(bargap=0.05)
fig_popularity.show()


In [36]:
fig = px.box(
    unique_tracks_df,
    x='Content Type',
    y='track_popularity',
    color='Content Type',
    color_discrete_map={'Non-Explicit': '#1f77b4', 'Explicit': '#d62728'}, # Matching our previous color scheme
    title='Distribution of Track Popularity: Explicit vs. Non-Explicit',
    labels={
        'track_popularity': 'Popularity Score', 
        'Content Type': 'Content Type'
    }
)

fig.update_layout(
    title_font=dict(size=18, family="Arial", color="black"),
    title_x=0.5,           # Centers the title
    showlegend=False,      # Hides the legend since the x-axis already explains the colors
    plot_bgcolor='white',  # Clean white background
    width=800,             # Adjusts the width of the figure
    height=600
)

fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_xaxes(showgrid=False)

fig.show()

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['track_id', 'artists', 'album_name', 'track_name', 'track_popularity', 'duration_ms', 'release_date', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre', 'duration_min', 'release_date_parsed', 'release_year', 'artist_name', 'artist_id', 'followers', 'artist_genres', 'artist_popularity'] but received: Content Type

In [37]:
genre_summary = (
    tracks_clean
    .groupby("track_genre")
    .agg(
        n_tracks=("track_id", "count"),
        avg_popularity=("track_popularity", "mean"),
        median_duration_min=("duration_min", "median"),
        avg_danceability=("danceability", "mean"),
        avg_energy=("energy", "mean"),
        avg_tempo=("tempo", "mean"),
        pct_explicit=("explicit", "mean"),
    )
    .sort_values("avg_popularity", ascending=False)
)

genre_summary.head(10).round(2)


,n_tracks,avg_popularity,median_duration_min,avg_danceability,avg_energy,avg_tempo,pct_explicit
track_genre,,,,,,,
pop-film,1000,59.28,4.67,0.60,0.60,116.79,0.00
k-pop,999,56.95,3.83,0.65,0.68,120.16,0.05
chill,1000,53.65,2.77,0.66,0.43,114.29,0.17
sad,1000,52.38,2.49,0.69,0.46,118.77,0.45
grunge,1000,49.59,3.83,0.46,0.80,130.21,0.07
indian,1000,49.54,3.96,0.59,0.57,116.32,0.02
anime,1000,48.77,3.59,0.54,0.67,124.82,0.06
emo,1000,48.13,3.08,0.60,0.67,127.61,0.46
sertanejo,1000,47.87,3.26,0.59,0.71,126.89,0.01


In [38]:
top_genres = ['hip hop', 'classical', 'country', 'electronic', 'rock']

genre_explicit_pivot = (
    tracks_clean[tracks_clean["track_genre"].isin(top_genres)]
    .pivot_table(
        index="track_genre",
        columns="explicit",
        values="track_popularity",
        aggfunc="mean",
    )
    .rename(columns={False: "non_explicit_avg_popularity", True: "explicit_avg_popularity"})
    .sort_values("non_explicit_avg_popularity", ascending=False)
)

genre_explicit_pivot.round(2)


explicit,non_explicit_avg_popularity,explicit_avg_popularity
track_genre,,
electronic,43.85,47.82
rock,18.63,27.35
country,16.29,40.97
classical,13.06,NaN


## Step 3: Assessment of Missingness

In [39]:
# We assess missingness on the RAW tempo column, since Step 2 imputes tempo.
tempo_missing = tracks_raw['tempo'].isna()

def diff_in_means(values, mask):
    '''|mean(values where mask) - mean(values where not mask)|  -- for a QUANTITATIVE column.'''
    return abs(values[mask].mean() - values[~mask].mean())

def tvd(codes, mask, n_categories):
    '''Total variation distance between the category distributions of the two groups,
    where `codes` are integer category labels (0..n_categories-1) -- for a CATEGORICAL column.'''
    p1 = np.bincount(codes[mask],  minlength=n_categories) / mask.sum()
    p0 = np.bincount(codes[~mask], minlength=n_categories) / (~mask).sum()
    return np.abs(p1 - p0).sum() / 2

def permutation_test(values, mask, stat_fn, n_perm=1000, seed=42, **kwargs):
    '''Shuffle the missingness indicator and build the null distribution of `stat_fn`.'''
    observed = stat_fn(values, mask, **kwargs)
    rng = np.random.default_rng(seed)
    shuffled_mask = mask.copy()
    null_stats = np.empty(n_perm)
    for i in range(n_perm):
        rng.shuffle(shuffled_mask)               # reshuffle which rows are "missing"
        null_stats[i] = stat_fn(values, shuffled_mask, **kwargs)
    p_value = float((null_stats >= observed).mean())
    return observed, p_value, null_stats

## Step 4: Hypothesis Testing

We test whether explicit and non-explicit tracks differ in popularity. We use the deduplicated `unique_tracks_df` so that each track is counted once (a track can appear under several genre labels).

- **Null hypothesis:** Explicit and non-explicit tracks have the same mean popularity; any observed difference is due to chance.
- **Alternative hypothesis:** Explicit and non-explicit tracks have different mean popularities.
- **Test statistic:** the absolute difference in mean `track_popularity` between the two groups, |mean(explicit) − mean(non-explicit)|. We use the *absolute* difference because the alternative is two-sided (a difference in either direction).
- **Significance level:** 0.05.

A permutation test is appropriate here: we are comparing a numerical outcome across two groups and make no assumption about the shape of the popularity distribution. Under the null the `explicit` label is exchangeable, so we approximate the null distribution by repeatedly shuffling it.

In [40]:
# Null: There is no difference in the mean popularity scores between explicit and non-explicit tracks.

# Alternative: There is a significant difference in the mean popularity scores between explicit and non-explicit tracks.

# Permutation test: do explicit and non-explicit tracks differ in mean popularity?
pop_values  = unique_tracks_df['track_popularity'].to_numpy()
is_explicit = unique_tracks_df['explicit'].to_numpy().astype(bool)

observed_diff = abs(pop_values[is_explicit].mean() - pop_values[~is_explicit].mean())

rng = np.random.default_rng(0)
shuffled = is_explicit.copy()
n_perm = 10_000
null_diffs = np.empty(n_perm)
for i in range(n_perm):
    rng.shuffle(shuffled)                     # shuffle the explicit labels under the null
    null_diffs[i] = abs(pop_values[shuffled].mean() - pop_values[~shuffled].mean())

p_value = float((null_diffs >= observed_diff).mean())

print(f"mean popularity  --  explicit: {pop_values[is_explicit].mean():.3f} (n={is_explicit.sum()})  |  "
      f"non-explicit: {pop_values[~is_explicit].mean():.3f} (n={(~is_explicit).sum()})")
print(f"observed |difference in means| = {observed_diff:.4f}")
print(f"permutation p-value = {p_value:.4f}")

mean popularity  --  explicit: 36.886 (n=7704)  |  non-explicit: 32.853 (n=82036)
observed |difference in means| = 4.0331
permutation p-value = 0.0000


In [41]:
# Empirical null distribution of the test statistic, with the observed value marked.
fig = px.histogram(
    x=null_diffs, nbins=40,
    title=('Permutation Null Distribution: Explicit vs. Non-Explicit Popularity'
           '<br><sup>Test statistic: |difference in mean popularity|</sup>'),
    labels={'x': '|difference in mean popularity|'},
)
fig.add_vline(x=observed_diff, line_color='red', line_dash='dash',
              annotation_text=f'observed = {observed_diff:.2f} (p = {p_value:.3f})',
              annotation_position='top left')
fig.update_layout(plot_bgcolor='white', title_x=0.5, showlegend=False, bargap=0.05)
fig.show()

**Conclusion.** Explicit tracks have a higher mean popularity (≈36.9) than non-explicit tracks (≈32.9) — an observed difference of ≈4.0 points. Across 10,000 permutations the shuffled difference never reached the observed value, giving a p-value of ≈0.000, far below our 0.05 threshold. We therefore **reject the null hypothesis**: the data are highly inconsistent with explicit and non-explicit tracks having the same mean popularity. Because this is an observational permutation test rather than a controlled experiment, this is evidence of an *association* — explicit tracks tend to be more popular in this catalog — not proof that explicit content *causes* higher popularity (genre is a plausible confounder; e.g., hip-hop is both frequently explicit and popular).

## Step 5: Framing a Prediction Problem

### Framing the Prediction Problem

**Problem.** We predict a track's **popularity score** (`track_popularity`, 0–100) from its audio features, metadata, and artist information. This is a **regression** problem, and it continues the popularity theme from our earlier analysis.

**Response variable.** `track_popularity`. We chose it because popularity is the central quantity our whole project investigates, and Spotify reports it on a meaningful 0–100 scale, so a continuous regression target is natural (rather than binarizing it and discarding information).

**Evaluation metric.** We use **RMSE** (root mean squared error) as the primary metric, reported alongside **R²**. RMSE is in the same units as the response (popularity points), so it is directly interpretable — "our predictions are off by about *X* points on average" — and squaring penalizes large misses, which matters when a model badly mis-ranks a track. We prefer RMSE over plain MAE because we care about avoiding large errors, and over a classification metric like accuracy because the target is continuous. R² is included to communicate how much of the variation in popularity the model explains.

**Genre selection.** Following the dataset's guidance to choose at least five musically distinct genres, we restrict the modeling data to eight — **pop, rock, hip-hop, classical, jazz, country, edm, and metal**. They are easily recognizable and span very different musical profiles (vocal-driven pop, acoustic/instrumental classical and jazz, electronic edm, aggressive metal, …) as well as very different popularity and explicit-content profiles, which makes both the model and the later fairness analysis (explicit vs. non-explicit) meaningful. A track can carry several genre labels in this dataset, so we keep **one row per track** (its first-listed genre) to avoid counting a track more than once and to prevent the same track from leaking across the train/test split.

**Time of prediction / avoiding leakage.** Imagine predicting a track's popularity at the moment it is released. At that point we *do* know its audio features (Spotify computes `danceability`, `energy`, `valence`, `tempo`, `loudness`, etc. from the recording), its `track_genre`, whether it is `explicit`, its duration, its release year, and the releasing artist's `followers` and artist-level `popularity` (which reflect the artist's standing from prior work). We do **not** know the track's own future popularity — that is the response. We therefore train only on the features below and exclude anything that would only be known after release.

In [42]:
from sklearn.model_selection import train_test_split

CHOSEN_GENRES = ['pop', 'rock', 'hip-hop', 'classical', 'jazz', 'country', 'edm', 'metal']

# Candidate features, all known at the time a track is released (no leakage).
AUDIO_FEATURES  = ['danceability', 'energy', 'valence', 'speechiness', 'acousticness',
                   'instrumentalness', 'liveness', 'loudness', 'tempo', 'duration_min']
OTHER_FEATURES  = ['release_year', 'explicit', 'track_genre', 'key', 'mode', 'time_signature']
ARTIST_FEATURES = ['followers', 'artist_popularity']   # ~1% missing here -> imputed in the final model
FEATURES = AUDIO_FEATURES + OTHER_FEATURES + ARTIST_FEATURES
RESPONSE = 'track_popularity'

# One row per track, restricted to the chosen genres.
model_df = (
    unique_tracks_df[unique_tracks_df['track_genre'].isin(CHOSEN_GENRES)]
    [FEATURES + [RESPONSE]]
    .reset_index(drop=True)
)

# A single train/test split, reused UNCHANGED in Step 6 (Baseline) and Step 7 (Final),
# so the two models are compared on the same held-out data.
X_train, X_test, y_train, y_test = train_test_split(
    model_df[FEATURES], model_df[RESPONSE], test_size=0.25, random_state=42
)

print("modeling rows:", len(model_df), "across", model_df['track_genre'].nunique(), "genres")
print("train:", X_train.shape, "| test:", X_test.shape)
print("explicit tracks in test set:", int(X_test['explicit'].sum()))
model_df['track_genre'].value_counts()

modeling rows: 4863 across 8 genres
train: (3647, 18) | test: (1216, 18)
explicit tracks in test set: 118


track_genre
country      946
classical    867
hip-hop      842
edm          694
jazz         524
pop          416
rock         342
metal        232
Name: count, dtype: int64

## Step 6: Baseline Model

### Baseline Model

Our baseline is a simple **linear regression** that predicts `track_popularity` from **two quantitative audio features**: `danceability` and `energy`. Both are already on a 0–1 scale and need no encoding (0 ordinal, 0 nominal features), so the whole model is a one-step `Pipeline` containing only the estimator. We evaluate on the 25% test split created in Step 5 to measure how well it generalizes.

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))   # version-safe RMSE

baseline_features = ['danceability', 'energy']

baseline_model = Pipeline([
    ('model', LinearRegression()),
])
baseline_model.fit(X_train[baseline_features], y_train)

baseline_pred = baseline_model.predict(X_test[baseline_features])
baseline_rmse = rmse(y_test, baseline_pred)
baseline_r2   = r2_score(y_test, baseline_pred)

print(f"Baseline (LinearRegression on {baseline_features}):")
print(f"  test RMSE = {baseline_rmse:.3f}")
print(f"  test R^2  = {baseline_r2:.3f}")

Baseline (LinearRegression on ['danceability', 'energy']):
  test RMSE = 29.528
  test R^2  = 0.110


**Is this model good?** Not really — which is expected for a baseline. Its test RMSE is ≈29.5 popularity points and its R² is ≈0.11, so the two audio features explain only about 11% of the variation in popularity. That is only modestly better than predicting the average popularity for every track (RMSE ≈31.3). This gives us a clear bar to beat in the final model.

## Step 7: Final Model

### Final Model

The final model is a **`RandomForestRegressor`** trained on a much richer feature set, all of it known at the time a track is released:

- **Audio features** (quantitative): the ten features `danceability` … `duration_min`.
- **`release_year`** (quantitative) and **`explicit`** (binary).
- **Categorical features**, one-hot encoded: `track_genre`, `key`, `mode`, `time_signature`.
- **Artist features** (quantitative): `followers` and `artist_popularity`, median-imputed for the ~1% of tracks with no artist match.
- **Two engineered features** (quantitative), beyond the baseline's columns:
  - **`n_artists`** — the number of collaborating artists on the track (parsed from the `artists` field). Collaborations and features tend to broaden a track's reach, so this should carry popularity signal that no single audio feature does.
  - **`artist_genre_breadth`** — how many genre tags the artist carries. Artists who span many genres are typically more established and widely listened to, which we expect to relate to popularity. Both are computed per-track from columns we already have, so they introduce no leakage.

We chose a random forest because the relationship between these features and popularity is clearly nonlinear and interaction-heavy (the baseline's linear fit captured little), and trees handle mixed feature types and interactions automatically.

**Hyperparameters to tune (chosen before tuning).** Using `GridSearchCV` (3-fold) we search over:
- **`max_depth`** ∈ {None, 15, 25} — the main control on model complexity: shallow trees underfit, unlimited-depth trees overfit, so cross-validation picks the depth that generalizes best.
- **`min_samples_leaf`** ∈ {1, 5} — a regularizer: requiring more samples per leaf smooths predictions and guards against overfitting.

We hold the number of trees at 200 (more trees mainly add stability, not bias). All preprocessing and the model live in a single `Pipeline`, fit only on the training data and evaluated on the same held-out test set as the baseline.

In [44]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Engineer the two new features and reuse the EXACT train/test rows from Step 5.
model_df_final = (
    unique_tracks_df[unique_tracks_df['track_genre'].isin(CHOSEN_GENRES)]
    .assign(
        n_artists=lambda df: df['artists'].str.count(';') + 1,
        artist_genre_breadth=lambda df: df['artist_genres'].apply(
            lambda v: len(v) if isinstance(v, list) else np.nan),
        explicit=lambda df: df['explicit'].astype(int),
        release_year=lambda df: df['release_year'].astype('float'),
    )
    [FEATURES + ['n_artists', 'artist_genre_breadth', RESPONSE]]
    .reset_index(drop=True)
)
FINAL_FEATURES = FEATURES + ['n_artists', 'artist_genre_breadth']
X_train_final = model_df_final.loc[X_train.index, FINAL_FEATURES]   # same rows as the baseline
X_test_final  = model_df_final.loc[X_test.index,  FINAL_FEATURES]

numeric_features     = AUDIO_FEATURES + ['release_year', 'n_artists', 'artist_genre_breadth'] + ARTIST_FEATURES
categorical_features = ['track_genre', 'key', 'mode', 'time_signature']

preprocessor = ColumnTransformer([
    ('num',  SimpleImputer(strategy='median'),       numeric_features),
    ('expl', 'passthrough',                          ['explicit']),
    ('cat',  OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

final_pipeline = Pipeline([
    ('pre',   preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    'model__max_depth':        [None, 15, 25],
    'model__min_samples_leaf': [1, 5],
    'model__n_estimators':     [200],
}
search = GridSearchCV(final_pipeline, param_grid, scoring='neg_root_mean_squared_error', cv=3, n_jobs=-1)
search.fit(X_train_final, y_train)

final_model = search.best_estimator_
final_pred  = final_model.predict(X_test_final)

print("best hyperparameters:", search.best_params_)
print(f"final    -> test RMSE = {rmse(y_test, final_pred):.3f}   R^2 = {r2_score(y_test, final_pred):.3f}")
print(f"baseline -> test RMSE = {baseline_rmse:.3f}   R^2 = {baseline_r2:.3f}")

best hyperparameters: {'model__max_depth': 15, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
final    -> test RMSE = 20.374   R^2 = 0.576
baseline -> test RMSE = 29.528   R^2 = 0.110


In [45]:
# Which features the final model relies on most.
ohe_names = (final_model.named_steps['pre']
             .named_transformers_['cat'].get_feature_names_out(categorical_features))
feature_names = numeric_features + ['explicit'] + list(ohe_names)
importances = (pd.Series(final_model.named_steps['model'].feature_importances_, index=feature_names)
               .sort_values(ascending=False).head(12))

fig = px.bar(
    importances.iloc[::-1], orientation='h',
    title='Final Model: Top 12 Feature Importances',
    labels={'value': 'importance', 'index': ''},
)
fig.update_layout(plot_bgcolor='white', title_x=0.5, showlegend=False)
fig.show()

**Improvement over the baseline.** On the *same* held-out tracks, the final model's test RMSE drops to ≈20.4 (from ≈29.5) and its R² rises to ≈0.58 (from ≈0.11), so the gain comes from the model and features, not an easier test set. The forest now explains the majority of the variation in popularity.

The feature-importance chart shows why: the two **artist features** (`artist_popularity` and `followers`) are among the strongest predictors, alongside audio characteristics like `energy` and `acousticness` and the track's `release_year`. This matches the data-generating process — popularity is driven heavily by *who* released a track and how recent it is, not by audio alone, which is exactly the signal the baseline lacked. The engineered `n_artists` and `artist_genre_breadth` features contribute smaller shares.

## Step 8: Fairness Analysis

### Fairness Analysis

We ask whether the final model is **less accurate for explicit tracks than for non-explicit tracks**, on the held-out test set.

- **Group X:** explicit tracks. **Group Y:** non-explicit tracks (binarizing the `explicit` column).
- **Evaluation metric:** **RMSE** within each group. This is a regression model, so classification metrics like precision do not apply; RMSE measures average prediction error in popularity points.
- **Null hypothesis:** the model is fair — its RMSE for explicit and non-explicit tracks is the same, and any difference is due to chance.
- **Alternative hypothesis:** the model is unfair — its RMSE differs between the two groups.
- **Test statistic:** the absolute difference in group RMSE, |RMSE(explicit) − RMSE(non-explicit)|.
- **Significance level:** 0.05.

We compute each track's squared error from the **already-fitted** final model, then permute the explicit/non-explicit labels to build the null distribution of the RMSE gap.

In [46]:
# Fairness permutation test: does the model's RMSE differ for explicit vs non-explicit tracks?
squared_errors   = (y_test.to_numpy() - final_pred) ** 2
is_explicit_test = (X_test_final['explicit'] == 1).to_numpy()

def group_rmse_pair(sq_err, mask):
    return np.sqrt(sq_err[mask].mean()), np.sqrt(sq_err[~mask].mean())

rmse_explicit, rmse_non_explicit = group_rmse_pair(squared_errors, is_explicit_test)
observed_gap = abs(rmse_explicit - rmse_non_explicit)

rng = np.random.default_rng(7)
shuffled = is_explicit_test.copy()
n_perm = 10_000
null_gaps = np.empty(n_perm)
for i in range(n_perm):
    rng.shuffle(shuffled)
    a, b = group_rmse_pair(squared_errors, shuffled)
    null_gaps[i] = abs(a - b)

p_value = float((null_gaps >= observed_gap).mean())

print(f"RMSE -- explicit: {rmse_explicit:.3f} (n={is_explicit_test.sum()})  |  "
      f"non-explicit: {rmse_non_explicit:.3f} (n={(~is_explicit_test).sum()})")
print(f"observed |RMSE difference| = {observed_gap:.4f}")
print(f"permutation p-value = {p_value:.4f}")

RMSE -- explicit: 26.197 (n=118)  |  non-explicit: 19.646 (n=1098)
observed |RMSE difference| = 6.5505
permutation p-value = 0.0013


In [47]:
# Null distribution of the RMSE gap, with the observed value marked.
fig = px.histogram(
    x=null_gaps, nbins=40,
    title=('Fairness Permutation Test: RMSE Gap, Explicit vs. Non-Explicit'
           '<br><sup>Test statistic: |RMSE(explicit) − RMSE(non-explicit)|</sup>'),
    labels={'x': '|difference in group RMSE|'},
)
fig.add_vline(x=observed_gap, line_color='red', line_dash='dash',
              annotation_text=f'observed = {observed_gap:.2f} (p = {p_value:.3f})',
              annotation_position='top left')
fig.update_layout(plot_bgcolor='white', title_x=0.5, showlegend=False, bargap=0.05)
fig.show()

**Conclusion.** The model's RMSE is ≈26.2 for explicit tracks versus ≈19.6 for non-explicit tracks — a gap of ≈6.6 popularity points. With a permutation p-value of ≈0.001 (well below 0.05), we **reject the null hypothesis**: such a gap is very unlikely under equal accuracy, so the model is **less accurate for explicit tracks**.

A plausible explanation lies in the data-generating process: explicit tracks are a small minority (~9% of the training data) and are concentrated in genres like hip-hop where popularity is high and highly variable, so the model has both less data and a harder target for this group. This is worth flagging — predictions should be treated with more caution for explicit tracks — and it points to next steps such as collecting more explicit examples or modeling that segment separately. As with any permutation test, this identifies a disparity in performance, not its ultimate cause.